# RETECO · судья ответов 2b и 2c

Оценивает каждый ответ из `reteco/gen` по пяти осям организаторов (1–5):
correctness, completeness, relevance, conversational coherence, faithfulness.
Плюс тип ответа (ответил / отказался / частично) и «есть ли утверждения без
опоры в документах». Судья видит историю, вопрос, документы, которые видел
ридер, эталонный ответ (`turns[].answer` — для оценки его брать можно) и ответ.

Организаторы судят GPT-4o с промптами, которые откроют только в январе. Здесь —
наша версия судьи. **Какой моделью судить — решает Иван** (docs/NOTES.md).
Два бэкенда:
- `BACKEND = "vllm"` — открытая модель здесь же (A100: Qwen3-14B, L4: Qwen3-8B,
  T4: Qwen3-8B-AWQ). Бесплатно, воспроизводимо.
- `BACKEND = "api"` — OpenAI-совместимый API, например `gpt-4o`, как у
  организаторов. Ключ — в секретах Colab (значок ключа слева): `OPENAI_API_KEY`,
  при другом провайдере ещё `OPENAI_BASE_URL`. GPU не нужен.

Судья обязан быть другой моделью, чем ридер: файлы того же ридера пропускаются.

**Как запускать.** После `generate.ipynb`. Выбрать бэкенд в ячейке 1 →
«Выполнить все». Оборвалось — ещё раз, готовые оценки пропускаются.

**Сколько.** ≈7 тыс. ответов. vLLM на A100 — 30–60 мин; API — зависит от
лимитов ключа, GPT-4o порядка $15–25 за всё.

**Что получится.** `reteco/judge/<судья>/<файл генерации>.jsonl` и сводка
`reteco/judge/<судья>/summary.csv`.

In [ ]:
# ── 1. Настройки ─────────────────────────────────────────────────────────────
BACKEND = "vllm"      # "vllm" или "api"
JUDGE = "auto"        # vllm: auto = по GPU; api: имя модели, например "gpt-4o"
GEN_GLOB = "*_dev.jsonl"   # какие файлы из Drive/reteco/gen судить
MAXLEN = 16384
CHUNK = 256           # оценок между записями на Drive
API_WORKERS = 8       # параллельных запросов к API
DRIVE = "/content/drive/MyDrive/reteco"
NEED_GPU = BACKEND == "vllm"

In [ ]:
# ── 2. Drive, vLLM, GPU ─────────────────────────────────────────────────────
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
from google.colab import drive
drive.mount("/content/drive")
!pip -q install vllm openai pytrec-eval-terrier 2>&1 | tail -2
import torch

# vllm ставит свой torch, а torchvision/torchaudio из Colab остаются под другую
# CUDA, и import vllm падает. Лечение из occ-rag-bfcl: torchaudio снести,
# torchvision поставить под CUDA нового torch. После правки нужен перезапуск.
def _imports_ok(mod):
    try:
        __import__(mod); return True
    except RuntimeError:
        return False       # стоит, но собран под другую CUDA
    except ImportError:
        return True        # не стоит — и конфликта нет
fixed = False
if not _imports_ok("torchaudio"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchaudio"], check=True)
    fixed = True
if not _imports_ok("torchvision") and "+" in torch.__version__:
    base, cu = torch.__version__.split("+")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--force-reinstall", "--no-deps",
                    f"torchvision==0.{int(base.split('.')[1]) + 15}.*",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"], check=True)
    fixed = True
if fixed:
    raise RuntimeError("Зависимости исправлены. Среда выполнения → Перезапустить сеанс, "
                       "потом снова «Выполнить все».")
if globals().get("NEED_GPU", True):
    assert torch.cuda.is_available(), "Нет GPU: Среда выполнения → Сменить среду выполнения → A100 или L4"
MEM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if MEM and torch.cuda.is_bf16_supported() else "half"   # T4 bf16 не умеет
BIG = MEM >= 39                                                             # A100 40/80 ГБ
print(torch.cuda.get_device_name(0) if MEM else "без GPU", f"{MEM:.0f} ГБ, {DTYPE}, torch", torch.__version__)

def load_llm(model, maxlen, util=0.88):
    """На время конструктора — настоящие stdout/stderr: в ядре Jupyter vLLM падает
    на sys.stdout.fileno() ещё до загрузки весов (occ-rag-bfcl)."""
    from vllm import LLM
    out, err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        return LLM(model=model, dtype=DTYPE, max_model_len=maxlen, gpu_memory_utilization=util, seed=0)
    finally:
        sys.stdout, sys.stderr = out, err

def chat(tok, msgs):
    """Промпт по шаблону модели; у Qwen3 размышления выключены (быстро, детерминированно)."""
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
# ── 3. Данные RETECO, трек 2 (≈270 МБ, минута) ──────────────────────────────
from huggingface_hub import snapshot_download
snapshot_download("DataScience-UIBK/RETECO-SemEval2027", repo_type="dataset",
                  local_dir="/content/reteco",
                  allow_patterns=["track2_recor/*/documents.jsonl",
                                  "track2_recor/*/benchmark_*.json",
                                  "track2_recor/*/qrels_*.txt"])

In [ ]:
# ── 4. Общий код. Повторяет scripts/retrieve.py и score.py из репозитория ──
import os, re, json, time, glob
import numpy as np, pytrec_eval
from tqdm.auto import tqdm

DATA = "/content/reteco/track2_recor"
DOMAINS = ["drones", "politics", "law", "medicalsciences", "hardware", "economics",
           "psychology", "biology", "sustainable_living", "robotics", "earth_science"]  # от малых к большим
# официальный BM25 «реплика + история», dev, nDCG@10 — для сравнения
BM25_DEV = {"biology": .6065, "drones": .2954, "earth_science": .6264, "economics": .4739,
            "hardware": .3378, "law": .3437, "medicalsciences": .1730, "politics": .3219,
            "psychology": .5726, "robotics": .5123, "sustainable_living": .5531}
BM25_MACRO = {"dev": 0.4379, "train": 0.4539}
RUNS, QUERIES = f"{DRIVE}/runs", f"{DRIVE}/queries"
os.makedirs(RUNS, exist_ok=True); os.makedirs(QUERIES, exist_ok=True)

def load_corpus(dom):
    ids, texts = [], []
    for ln in open(f"{DATA}/{dom}/documents.jsonl", encoding="utf-8"):
        if ln.strip():
            d = json.loads(ln); ids.append(d["doc_id"]); texts.append(d["content"])
    return ids, texts

def topics(dom, split, qset):
    """Реплики без золота пропускаются; история приклеивается как у организаторов.
    qset: "query", "hist" или имя папки в Drive/reteco/queries (варианты rewrite.py)."""
    given = None
    if qset not in ("query", "hist"):
        with open(f"{QUERIES}/{qset}/{split}.jsonl", encoding="utf-8") as f:
            # ключ (домен, topic_id): в доменах BRIGHT topic_id повторяются между доменами
            given = {(r["domain"], r["topic_id"]): r["text"] for r in map(json.loads, filter(str.strip, f))}
    ids, texts = [], []
    for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
        for t in c["turns"]:
            if not (t.get("gold_doc_ids") or t.get("supporting_doc_ids")):
                continue
            qid = f"{c['id']}_turn_{t['turn_id']}"
            if given is not None:
                text = given[(dom, qid)]
            else:
                text, h = t["query"], t.get("conversation_history", "")
                if qset == "hist" and h and h != "No previous conversation.":
                    text += f"\n\nConversation History:\n{h}"
            ids.append(qid); texts.append(text)
    return ids, texts

def ndcg10(run, dom, split):
    """run: {qid: {doc: score}}. Реплика без прогона получает 0, как на лидерборде."""
    qrels = {}
    for ln in open(f"{DATA}/{dom}/qrels_{split}.txt"):
        p = ln.split()
        if len(p) == 4:
            qrels.setdefault(p[0], {})[p[2]] = int(p[3])
    res = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"}).evaluate(run)
    return sum(v["ndcg_cut_10"] for v in res.values()) / len(qrels)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Сначала во временный файл, потом переименование: оборванная запись не
    оставляет полуфайл, который перезапуск принял бы за готовый."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            for r, (d, s) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                f.write(f"{q}\tQ0\t{d}\t{r}\t{s:.6f}\t{tag}\n")
    os.replace(path + ".tmp", path)

In [ ]:
%%writefile /content/gen_worker.py
"""Ответы одного ридера на все задания (2b и 2c по первым стадиям). Запускается
из generate.ipynb отдельным процессом на каждый ридер: так vLLM точно отдаёт
память GPU перед следующей моделью. Модуль же импортирует ячейка итогов."""
import json, math, os, re, sys, time

OCC_STOP = [151643, 151645, 151683]   # <|endoftext|> <|im_end|> <|answer_end|> — generation_config OCC
CONTROL_SYSTEM = (
    "You are a helpful assistant in a multi-turn conversation. Answer the user's latest message "
    "using only the numbered documents attached to it.\n"
    "Reply in this format:\n"
    "Line 1: ANSWERABLE if the documents contain enough information to answer, otherwise UNANSWERABLE.\n"
    "Next lines: if ANSWERABLE, the answer in 2-5 sentences in a natural conversational tone; "
    "if UNANSWERABLE, say briefly what information is missing.")


def load_turns(data, domains, split):
    """Реплики с золотом. history — прошлые реплики диалога (вопрос, ответ): это и есть
    conversation_history. reference — золотой ответ ТЕКУЩЕЙ реплики: только для аудита
    и судьи, в промпт ридера не идёт. Золото — только то, что есть в qrels: 3 золотых id
    в politics отсутствуют в корпусе, организаторы выкинули их из qrels."""
    out = []
    for dom in domains:
        qrels = {tuple(l.split()[0:3:2]) for l in open(f"{data}/{dom}/qrels_{split}.txt") if l.strip()}
        for c in json.load(open(f"{data}/{dom}/benchmark_{split}.json", encoding="utf-8")):
            prev = []
            for t in c["turns"]:
                qid = f"{c['id']}_turn_{t['turn_id']}"
                gold = [d for d in t.get("gold_doc_ids") or t.get("supporting_doc_ids") or [] if (qid, d) in qrels]
                if gold:
                    out.append({"domain": dom, "topic_id": qid,
                                "turn_id": t["turn_id"], "query": t["query"], "history": list(prev),
                                "gold": list(gold), "reference": t["answer"]})
                prev.append((t["query"], t["answer"]))
    return out


def read_top(path, k):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], []).append((int(p[3]), p[2]))
    return {q: [d for _, d in sorted(v)[:k]] for q, v in run.items()}


def contexts(C, turns, split, job):
    """"2b" → золотые документы; "2c-<прогон>" → топ-K из Drive/reteco/runs/<прогон>_<split>."""
    if job == "2b":
        return {(t["domain"], t["topic_id"]): t["gold"] for t in turns}
    tops = {dom: read_top(f"{C['drive']}/runs/{job[3:]}_{split}/{dom}/run.trec", C["k_ctx"])
            for dom in C["domains"]}
    return {(t["domain"], t["topic_id"]): tops[t["domain"]].get(t["topic_id"], []) for t in turns}


def sufficiency(gold, ctx):
    """Сколько золота в контексте и метка: all — всё золото, что влезает в K; none — ни одного."""
    n = len(set(gold) & set(ctx))
    full = min(len(gold), len(ctx)) if ctx else len(gold)
    return n, ("none" if n == 0 else "all" if n >= full else "part")


def corpus_texts(data, need):
    """need: {домен: {doc_id}} → {(домен, doc_id): текст}."""
    out = {}
    for dom, ids in need.items():
        for ln in open(f"{data}/{dom}/documents.jsonl", encoding="utf-8"):
            if ln.strip():
                d = json.loads(ln)
                if d["doc_id"] in ids:
                    out[(dom, d["doc_id"])] = d["content"]
    return out


def history_msgs(t):
    m = []
    for q, a in t["history"]:
        m += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return m


class OCC:
    """Шаблон OCC сам кладёт документы в последнюю реплику user; генерация начинается
    с <|query_analysis_start|>, статус — <|status_start|>(UN)ANSWERABLE<|status_end|>."""
    stop, skip_special, marker = OCC_STOP, False, "<|status_start|>"
    STATUS = re.compile(r"<\|status_start\|>(.*?)(?:<\|status_end\|>|$)", re.S)
    ANSWER = re.compile(r"<\|answer_start\|>(.*?)(?:<\|answer_end\|>|<\|im_end\|>|$)", re.S)
    SPECIAL = re.compile(r"<\|[a-z_]+\|>")

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        msgs = history_msgs(t) + [{"role": "user", "content": t["query"]}]
        return self.tok.apply_chat_template(msgs, documents=[{"text": d} for d in docs],
                                            add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        m = self.STATUS.search(raw)
        status = self.SPECIAL.sub("", m.group(1)).strip().upper() if m else ""
        m = self.ANSWER.search(raw)
        return status, (self.SPECIAL.sub("", m.group(1)).strip() if m else "")

    def prefix(self, raw):
        i = raw.find(self.marker)
        return None if i < 0 else raw[:i + len(self.marker)]


class Control:
    """Открытая модель-контроль: тот же вход, статус первой строкой."""
    stop, skip_special = None, True

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        numbered = "\n\n".join(f"[{i}] {d}" for i, d in enumerate(docs, 1))
        msgs = ([{"role": "system", "content": CONTROL_SYSTEM}] + history_msgs(t)
                + [{"role": "user", "content": f"Documents:\n{numbered}\n\nQuestion: {t['query']}"}])
        return self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        first, _, rest = raw.strip().partition("\n")
        m = re.search(r"\b(UN)?ANSWERABLE\b", first.upper())
        if not m:
            return "", raw.strip()
        return m.group(0), rest.strip()

    def prefix(self, raw):
        return ""          # статус — самый первый токен ответа


def p_status(tok, lp):
    """Вероятность ANSWERABLE на позиции статуса: сумма вероятностей токенов-префиксов
    «ANSWERABLE» против «UNANSWERABLE» среди топ-20 (ANS… против UN…). mass — сколько
    вероятности пришлось на эти два варианта вообще."""
    a = u = 0.0
    for tid, x in lp.items():
        s = tok.decode([tid]).strip()
        if s and "UNANSWERABLE".startswith(s):
            u += math.exp(x.logprob)
        elif s and "ANSWERABLE".startswith(s):
            a += math.exp(x.logprob)
    return (round(a / (a + u), 6) if a + u > 0 else None), round(a + u, 6)


def main(reader_id, cfg_path):
    from vllm import LLM, SamplingParams
    C = json.load(open(cfg_path))
    llm = LLM(model=reader_id, dtype=C["dtype"], max_model_len=C["maxlen"],
              gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    R = (OCC if "OCC" in reader_id else Control)(tok)
    # OCC: только greedy, без repetition_penalty — иначе ломается структурный формат
    sp = SamplingParams(temperature=0.0, max_tokens=C["max_new"], stop_token_ids=R.stop,
                        skip_special_tokens=R.skip_special, spaces_between_special_tokens=False)
    sp1 = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
    short = reader_id.split("/")[-1]
    limit = C["maxlen"] - C["max_new"]
    ntok = lambda p: len(tok(p, add_special_tokens=False).input_ids)
    os.makedirs(f"{C['drive']}/gen", exist_ok=True)
    for split in C["splits"]:
        turns = load_turns(C["data"], C["domains"], split)
        jobs = [(j, contexts(C, turns, split, j)) for j in C["jobs"]]
        need = {}
        for _, ctx in jobs:
            for (dom, _), ids in ctx.items():
                need.setdefault(dom, set()).update(ids)
        text = corpus_texts(C["data"], need)
        for job, ctx in jobs:
            path = f"{C['drive']}/gen/{short}_{job}_{split}.jsonl"
            done = set()
            if os.path.exists(path):
                done = {(r["domain"], r["topic_id"]) for r in map(json.loads, open(path, encoding="utf-8"))}
            prompts, keep, skipped = [], [], 0
            for t in turns:
                if (t["domain"], t["topic_id"]) in done:
                    continue
                ids = ctx[(t["domain"], t["topic_id"])]
                docs = [text[(t["domain"], d)] for d in ids]
                p, trimmed = R.prompt(t, docs), False
                if ntok(p) > limit:      # не влезает — режем каждый документ, золото не выкидываем
                    p, trimmed = R.prompt(t, [" ".join(d.split()[:C["trim_words"]]) for d in docs]), True
                if ntok(p) > limit:
                    skipped += 1
                    continue
                prompts.append(p)
                keep.append((t, ids, trimmed))
            print(f"{short} · {job} · {split}: сделать {len(prompts)}, готово {len(done)}, "
                  f"не влезло {skipped}", flush=True)
            t0 = time.time()
            with open(path, "a", encoding="utf-8") as f:
                for s in range(0, len(prompts), C["chunk"]):
                    outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
                    raws = [o.outputs[0].text for o in outs]
                    # второй проход: промпт + ответ до места статуса, один токен, топ-20 логпробов
                    pref = [(i, R.prefix(r)) for i, r in enumerate(raws)]
                    pref = [(i, x) for i, x in pref if x is not None]
                    probs = {}
                    if pref:
                        o1 = llm.generate([prompts[s + i] + x for i, x in pref], sp1, use_tqdm=False)
                        for (i, _), o in zip(pref, o1):
                            probs[i] = p_status(tok, o.outputs[0].logprobs[0])
                    for i, (o, raw) in enumerate(zip(outs, raws)):
                        t, ids, trimmed = keep[s + i]
                        status, answer = R.parse(raw)
                        n_in, suff = sufficiency(t["gold"], ids)
                        p, mass = probs.get(i, (None, 0.0))
                        f.write(json.dumps({
                            "domain": t["domain"], "topic_id": t["topic_id"], "turn_id": t["turn_id"],
                            "split": split, "reader": reader_id, "mode": job[:2],
                            "first_stage": job[3:] or None, "context_ids": ids, "n_gold": len(t["gold"]),
                            "gold_in_ctx": n_in, "suff": suff, "status": status, "answer": answer,
                            "p_answerable": p, "p_mass": mass, "finish": o.outputs[0].finish_reason,
                            "trimmed": trimmed, "raw": raw}, ensure_ascii=False) + "\n")
                    f.flush()
                    print(f"  {s + len(outs)}/{len(prompts)} · {(time.time() - t0) / 60:.1f} мин", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

In [ ]:
# ── 6. Промпт судьи и разбор ответа ─────────────────────────────────────────
sys.path.insert(0, "/content")
import gen_worker as G
AXES = ["correctness", "completeness", "relevance", "coherence", "faithfulness"]
SYSTEM = ("You are an impartial expert evaluator of conversational question answering systems. "
          "You grade one assistant response at a time, strictly and consistently.")
RUBRIC = """Score each criterion from 1 (very poor) to 5 (excellent):
- correctness: factual agreement with the reference answer; wrong or contradicting facts lower it.
- completeness: how many key points of the reference answer the response covers.
- relevance: the response addresses the latest question and stays on topic.
- coherence: the response fits the conversation as the next turn - consistent with earlier turns, resolves references naturally.
- faithfulness: every claim is supported by the provided documents; a refusal that makes no claims gets 5.
Also classify:
- response_type: "answered" (a substantive answer), "refused" (declines or says the information is not available), or "partial" (answers part and says the rest is not available).
- unsupported_claims: true if the response states any fact not supported by the provided documents.

Return only a JSON object with keys in this order: "rationale" (at most 3 sentences), "correctness", "completeness", "relevance", "coherence", "faithfulness", "response_type", "unsupported_claims"."""
REFUSAL = "I can't answer this from the provided documents."

def judge_msgs(t, docs, rec):
    hist = "\n".join(f"User: {q}\nAssistant: {a}" for q, a in t["history"]) or "(none)"
    numbered = "\n\n".join(f"[{i}] {d}" for i, d in enumerate(docs, 1)) or "(none)"
    ans = rec["answer"] or (REFUSAL if rec["status"].startswith("UN") else "(empty)")
    user = (f"Grade the RESPONSE to the user's latest question in the conversation.\n\n"
            f"## Conversation so far\n{hist}\n\n## Latest question\n{t['query']}\n\n"
            f"## Documents the assistant was given\n{numbered}\n\n"
            f"## Reference answer (written by an expert who had the correct documents)\n{t['reference']}\n\n"
            f"## Response to grade\n{ans}\n\n{RUBRIC}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def parse_judge(raw):
    m = re.search(r"\{.*\}", raw or "", re.S)
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    out = {}
    for k in AXES:
        v = d.get(k)
        if isinstance(v, str) and v.strip().isdigit():
            v = int(v)
        if not isinstance(v, (int, float)) or not 1 <= v <= 5:
            return None
        out[k] = int(round(v))
    rt = str(d.get("response_type", "")).strip().lower()
    uc = d.get("unsupported_claims")
    uc = uc.strip().lower() == "true" if isinstance(uc, str) else uc
    if rt not in ("answered", "refused", "partial") or not isinstance(uc, bool):
        return None
    return {**out, "response_type": rt, "unsupported_claims": uc, "rationale": str(d.get("rationale", ""))[:1000]}

In [ ]:
# ── 7. Бэкенд: vLLM здесь же или API по ключу ───────────────────────────────
if BACKEND == "vllm":
    from vllm import SamplingParams
    if JUDGE == "auto":
        JUDGE = "Qwen/Qwen3-14B" if BIG else "Qwen/Qwen3-8B" if MEM >= 22 else "Qwen/Qwen3-8B-AWQ"
    llm = load_llm(JUDGE, MAXLEN)
    tok = llm.get_tokenizer()
    SP = SamplingParams(temperature=0.0, max_tokens=600)
    def run_batch(msg_lists):
        outs = llm.generate([chat(tok, m) for m in msg_lists], SP, use_tqdm=False)
        return [o.outputs[0].text for o in outs]
else:
    from openai import OpenAI
    from google.colab import userdata
    from concurrent.futures import ThreadPoolExecutor
    assert JUDGE != "auto", "для api указать модель в JUDGE, например gpt-4o"
    def secret(k):
        try:
            return userdata.get(k)
        except Exception:
            return None
    client = OpenAI(api_key=secret("OPENAI_API_KEY"), base_url=secret("OPENAI_BASE_URL") or None)
    def call(msgs):
        for attempt in range(6):
            try:
                r = client.chat.completions.create(model=JUDGE, messages=msgs, temperature=0, max_tokens=600,
                                                   response_format={"type": "json_object"})
                return r.choices[0].message.content or ""
            except Exception as e:
                print("  API:", type(e).__name__, str(e)[:120]); time.sleep(2 ** attempt)
        return ""
    def run_batch(msg_lists):
        with ThreadPoolExecutor(API_WORKERS) as ex:
            return list(ex.map(call, msg_lists))
SHORT = JUDGE.split("/")[-1]
OUTD = f"{DRIVE}/judge/{SHORT}"
os.makedirs(OUTD, exist_ok=True)
print("судья:", JUDGE, "→", OUTD)

In [ ]:
# ── 8. Оценка всех файлов генерации → Drive ─────────────────────────────────
turns = {}
for path in sorted(glob.glob(f"{DRIVE}/gen/{GEN_GLOB}")):
    recs = [json.loads(l) for l in open(path, encoding="utf-8")]
    if not recs:
        continue
    name = os.path.basename(path)
    if recs[0]["reader"].split("/")[-1] == SHORT:
        print(f"[пропуск] {name}: ридер и судья — одна модель"); continue
    split = recs[0]["split"]
    if split not in turns:
        turns[split] = {(t["domain"], t["topic_id"]): t for t in G.load_turns(DATA, DOMAINS, split)}
    out = f"{OUTD}/{name}"
    done = {(r["domain"], r["topic_id"]) for r in map(json.loads, open(out, encoding="utf-8"))} if os.path.exists(out) else set()
    todo = [r for r in recs if (r["domain"], r["topic_id"]) not in done and (r["domain"], r["topic_id"]) in turns[split]]
    need = {}
    for r in todo:
        need.setdefault(r["domain"], set()).update(r["context_ids"])
    text = G.corpus_texts(DATA, need)
    t0 = time.time()
    with open(out, "a", encoding="utf-8") as f:
        for s in range(0, len(todo), CHUNK):
            part = todo[s:s + CHUNK]
            msgs = [judge_msgs(turns[split][(r["domain"], r["topic_id"])],
                               [text[(r["domain"], d)] for d in r["context_ids"]], r) for r in part]
            raws = run_batch(msgs)
            bad = [i for i, x in enumerate(raws) if parse_judge(x) is None]
            if bad:          # одна повторная попытка с напоминанием про JSON
                again = run_batch([m[:-1] + [{"role": "user", "content": m[-1]["content"]
                                   + "\n\nReturn ONLY the JSON object, nothing else."}] for m in (msgs[i] for i in bad)])
                for i, x in zip(bad, again):
                    raws[i] = x
            for r, raw in zip(part, raws):
                j = parse_judge(raw)
                f.write(json.dumps({"domain": r["domain"], "topic_id": r["topic_id"], "split": split,
                                    "reader": r["reader"], "mode": r["mode"], "first_stage": r["first_stage"],
                                    "suff": r["suff"], "status": r["status"], "judge": JUDGE,
                                    "ok": j is not None, **(j or {}), "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  {name}: {s + len(part)}/{len(todo)} · {(time.time() - t0) / 60:.1f} мин")
    print(f"{name}: готово")

In [ ]:
# ── 9. Сводка: средние по осям, отказы и выдумки; 2c — по достаточности ─────
import csv
rows = []
for path in sorted(glob.glob(f"{OUTD}/*.jsonl")):
    rs = [json.loads(l) for l in open(path, encoding="utf-8")]
    good = [r for r in rs if r["ok"]]
    for suff in ("все", "all", "part", "none"):
        g = good if suff == "все" else [r for r in good if r["suff"] == suff]
        if not g or (suff != "все" and rs[0]["mode"] == "2b"):
            continue
        n = len(g)
        rows.append({"file": os.path.basename(path)[:-6], "suff": suff, "n": n,
                     **{a: round(sum(r[a] for r in g) / n, 3) for a in AXES},
                     "refused": round(sum(r["response_type"] == "refused" for r in g) / n, 3),
                     "unsupported": round(sum(r["unsupported_claims"] for r in g) / n, 3),
                     "parse_fail": sum(not r["ok"] for r in rs) if suff == "все" else ""})
assert rows, f"в {OUTD} нет разобранных оценок — смотреть поле raw в файлах"
with open(f"{OUTD}/summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print(f"{'файл':<42}{'дост.':>6}{'n':>5}" + "".join(f"{a[:5]:>7}" for a in AXES) + f"{'отказ':>7}{'выдум':>7}")
for r in rows:
    print(f"{r['file']:<42}{r['suff']:>6}{r['n']:>5}" + "".join(f"{r[a]:>7.2f}" for a in AXES)
          + f"{r['refused']:>7.2f}{r['unsupported']:>7.2f}")
print("\nвыдум — доля ответов с утверждениями без опоры в документах. Сводка:", f"{OUTD}/summary.csv")